[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shravani018/llm-audit-bench/blob/main/notebooks/07_aggregate_scores.ipynb)

#### 07 - Aggregate Scores 

Aggregates all 5 pillar scores (transparency, fairness, robustness, explainability, privacy) into a final trustworthiness index.

In [1]:
# Importing necessary libraries
import json
import os
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [4]:
def load(path, key):
    with open(path) as f:
        return {r['model_id']: r for r in json.load(f)[key]}
transparency   = load('../outputs/transparency_scores.json','transparency')
fairness       = load('../outputs/fairness_scores.json','fairness')
robustness     = load('../outputs/robustness_scores.json','robustness')
explainability = load('../outputs/explainability_scores.json','explainability')
privacy        = load('../outputs/privacy_scores.json','privacy')
models = list(transparency.keys())
print(f'loaded scores for {len(models)} models: {models}')

loaded scores for 5 models: ['gpt2', 'distilgpt2', 'facebook/opt-125m', 'EleutherAI/gpt-neo-125m', 'bigscience/bloom-560m']


In [6]:
weights = {
    'transparency':   0.15,
    'fairness':       0.25,
    'robustness':     0.25,
    'explainability': 0.20,
    'privacy':        0.15,
}
records = []
for model_id in models:
    t = transparency[model_id]['score']
    f = fairness[model_id]['fairness_score']
    r = robustness[model_id]['robustness_score']
    e = explainability[model_id]['explainability_score']
    p = privacy[model_id]['privacy_score']
    trust = round(
        weights['transparency']   * t +
        weights['fairness']       * f +
        weights['robustness']     * r +
        weights['explainability'] * e +
        weights['privacy']        * p,
        4
    )

    records.append({
        'model_id':        model_id,
        'transparency':    t,
        'fairness':        f,
        'robustness':      r,
        'explainability':  e,
        'privacy':         p,
        'trustworthiness': trust,
    })
df = pd.DataFrame(records).sort_values('trustworthiness', ascending=False).reset_index(drop=True)
df.index += 1
print(df[['model_id', 'trustworthiness']].to_string())

                  model_id  trustworthiness
1               distilgpt2           0.6063
2                     gpt2           0.5818
3  EleutherAI/gpt-neo-125m           0.5741
4        facebook/opt-125m           0.5314
5    bigscience/bloom-560m           0.5102


In [9]:
os.makedirs('../outputs', exist_ok=True)
output = {
    'weights': weights,
    'models':  df.to_dict(orient='records'),
}
with open('../outputs/aggregate_scores.json', 'w') as f:
    json.dump(output, f, indent=2)
print('saved to outputs/aggregate_scores.json')
print(df[['model_id', 'trustworthiness']].to_string())

saved to outputs/aggregate_scores.json
                  model_id  trustworthiness
1               distilgpt2           0.6063
2                     gpt2           0.5818
3  EleutherAI/gpt-neo-125m           0.5741
4        facebook/opt-125m           0.5314
5    bigscience/bloom-560m           0.5102
